# Gradient descent algorithm

This is a companion notebook code to demonstrate Gradient descent algorithm discussed in my blog post. Here I plot graphs and showcase the impact of batch size on the algorithm.

### Setup

We construct a synthetic data distribution: $y = $f(x) = 3x + 5x^2 + \epsilon$$ with some noise.

- we draw N=1000 samples from this ditribution
- and simulate noise with $\epsilon=10^{-4}$

and construct a Hypothesis function $$h_{\theta}(\theta, X) = X\theta$$
where, 
- $X \in \R^2$ is the input
- $y \in \R$ are the labels
- $\theta \in R^2$ vector are the randomly initialised model parameters.

This means, after training our hypothesis function should resolve

$$
\hat{\theta} = \begin{bmatrix} 3\\ 5 \end{bmatrix}
$$

and take MSE loss function: 

$$
    MSE_{\theta} = \frac{1}{N}\sum_{i=1}^N (y_i - h_{\theta}(x_i))^2
$$

to compute the gradients.

### Goal

Is to fit the hypothesis function to synthetic data distribution using Gradient descent with different batch sizes. 

In [134]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split

SEED = 0
np.random.seed(SEED)

PLOTS_DIR = Path("./plots")
PLOTS_DIR.mkdir(exist_ok=True)

### Quadratic function

As mentioned above we implement the following quadratic function with capability of tweaking $\epsilon$ 

In [125]:
class PolynomialFunction:
    def __init__(self, degree, coeffs=None, eps=1e-3):
        self.degree = degree
        self.eps = eps
        if coeffs is not None:
            self.coeffs = coeffs
        else:
            self.coeffs = np.random.rand(degree, 1)
        self.f = lambda x: x @ self.coeffs

    def __call__(self, x):
        return self.f(x) + np.random.randn()*self.eps
    
    def __str__(self):
        return f"PolynomialFunction(degree={self.degree}, theta={self.coeffs}, eps={self.eps})"

def test_quadratic_function():
    f = PolynomialFunction(degree=2, coeffs=np.array([0.12203823, 0.49517691]))
    exp_val = 4.*0.12203823 + 2.*0.49517691
    act_val = f(np.array([4., 2.]))
    assert np.allclose(exp_val, act_val, rtol=1e-02), f"Value {exp_val} != {act_val}"

test_quadratic_function()

### Hypothesis function implementatino

In [126]:
class Hypothesis:
    def __init__(self, degree, theta=None, track_params=False):
        super().__init__()
        self.trajectory = []
        self.degree = degree
        self.track_params = track_params
        if theta is not None:
            self.theta = theta
        else:
            self.theta = np.random.rand(degree, 1)
        
        self.track()

    def __call__(self, X):
        return X @ self.theta

    def __str__(self):
        return f"Hypothesis(degree={self.degree}, theta={self.theta})"

    def update(self, grad: np.array):
        self.theta -= grad
        self.track()
    
    def track(self):
        if self.track_params:
            self.trajectory.append(np.copy(self.theta))
    
def test_hypothesis():
    theta = np.array([1., 2.])
    h = Hypothesis(degree=2, theta=theta)
    exp_val = np.array([[1.*1. + 2.*2., 1.*4. + 2.*2.]])
    act_val = h(np.array([[1., 2.],[4., 2.]]))
    assert np.allclose(exp_val, act_val, rtol=1e-2), f"{exp_val} != {act_val}"

test_hypothesis()


### MSE loss

Here we implement the gradient descent algorithm in the most basic form. 
Let's first see how we can compute gradient using the loss function. 

$$
    \nabla_{\theta} MSE_{\theta} = \nabla_{\theta} \frac{1}{N}\sum_{i=1}^N (y_i - h_{\theta}(x_i))^2\\
    = \frac{1}{N} \sum_{i=1}^N \nabla_{\theta} (y_i - h_{\theta}(x_i))^2\\
    = - \frac{2}{N} \sum_{i=1}^N (y_i - h_{\theta}(x_i)) \cdot \nabla_{\theta} h_{\theta}(x_i)
$$

Since, 
$$
    h_{\theta}(\theta, x) = \theta^T x
$$

Taking the gradient wrt $\theta$ we get,

$$
    \nabla_{\theta} h_{\theta}(\theta, x) = x
$$

Therefore,
$$
    \nabla_{\theta} MSE_{\theta} = - \frac{2}{N} \sum_{i=1}^N (y_i - h_{\theta}(x_i)) x_i\\
    \qquad (\text{2 is just a constant that can be dropped})\\
    = - \frac{1}{N} \sum_{i=1}^N (y_i - h_{\theta}(x_i)) x_i
$$

This means that in order to compute the gradient we have following algorithm:

In [127]:
def gradient(y_pred: np.array, y_true: np.array, x: np.array):
    return -np.mean((y_true - y_pred) * x, axis=0)

def test_gradient():
    y_true = np.random.rand(1, 2).T
    y_pred = np.random.rand(1, 2).T
    x = np.random.rand(1, 2).T
    print(gradient(y_pred, y_true, x))

test_gradient()

[0.0058516]


### Dataset

In [128]:
np.repeat(np.array([1., 2., 3.]).reshape(-1, 1), repeats=3, axis=1) ** np.arange(1, 4)

array([[ 1.,  1.,  1.],
       [ 2.,  4.,  8.],
       [ 3.,  9., 27.]])

In [129]:
np.array([[1., 2., 3.],[4., 5., 6.]]) **np.arange(1, 4)

array([[  1.,   4.,  27.],
       [  4.,  25., 216.]])

In [130]:
class Dataset:

    def __init__(self, data, batch_size):
        self.index = 0
        self.batches = np.array_split(data, range(batch_size, data.shape[0], batch_size), axis=0)

    def __iter__(self):
        return self

    def __next__(self):
        if self.index < len(self.batches):
            batch = self.batches[self.index]
            self.index += 1
            return batch
        
        raise StopIteration

class TrainTestDataset:

    def __init__(self, func, test_size=0.2, batch_size=16, num=10000, eps=1e-3):
        self.batch_size = batch_size
        self.func = func
        degree = func.degree
        exp = np.arange(1, degree+1)
        X = np.linspace(-num, num, num=2*num+1)
        X = np.repeat(X.reshape(-1, 1), repeats=degree, axis=1) ** np.arange(1, degree+1)
        X = (X - np.mean(X, axis=0, keepdims=True)) / np.std(X, axis=0, keepdims=True)
        y = self.func(X)[..., None]

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X, y, test_size=test_size, random_state=SEED)
    
    def get_train_ds(self):
        return zip(Dataset(self.X_train, self.batch_size), Dataset(self.y_train, self.batch_size))
    
    def get_test_ds(self):
        return zip(Dataset(self.X_test, self.batch_size), Dataset(self.y_test, self.batch_size))

def validate_train_test_dataset():
    ds = TrainTestDataset(func=PolynomialFunction(degree=2), batch_size=4, num=10)
    train_ds = ds.get_train_ds()
    for X, y in train_ds:
        print(X, y)

    test_ds = ds.get_test_ds()
    for X, y in test_ds:
        print(X, y)

validate_train_test_dataset()


[[ 0.         -1.12186507]
 [ 0.66057826 -0.63232395]
 [ 1.32115652  0.83629942]
 [-0.66057826 -0.63232395]] [[[-0.42976949]]

 [[ 0.39451557]]

 [[ 1.59422141]]

 [[-0.87863377]]]
[[ 1.48630108  1.35643686]
 [-0.99086739 -0.02039755]
 [-1.32115652  0.83629942]
 [-0.82572282 -0.35695707]] [[[ 1.95280737]]

 [[-0.96228311]]

 [[-0.95207726]]

 [[-0.93219034]]]
[[ 0.99086739 -0.02039755]
 [-0.16514456 -1.09126875]
 [-0.49543369 -0.84649819]
 [ 1.15601195  0.37735461]] [[[ 0.94744089]]

 [[-0.57718126]]

 [[-0.8016134 ]]

 [[ 1.25909925]]]
[[-1.15601195  0.37735461]
 [-1.65144565  1.93776694]
 [ 0.82572282 -0.35695707]
 [ 0.33028913 -0.99947979]] [[[-0.96891209]]

 [[-0.84801622]]

 [[ 0.65924633]]

 [[-0.06455456]]]
[[-0.33028913 -0.99947979]
 [ 0.49543369 -0.84649819]
 [ 1.65144565  1.93776694]
 [-1.48630108  1.35643686]] [[[-0.70112923]]

 [[ 0.15324861]]

 [[ 2.33485713]]

 [[-0.91177864]]]
[[ 0.16514456 -1.09126875]] [[[-0.25889392]]]


### Training loop

In [131]:
import plotly.graph_objects as go

class Trainer:

    def __init__(self, model, ds, fig, epochs=1000, lr=1e-4):
        self.model = model
        self.ds = ds
        self.lr = lr
        self.epochs = epochs
        self.fig = fig

    def mse_loss(self, y_hat, y):
        return np.mean((y - y_hat)**2, axis=0)


    def train(self):
        train_losses, test_losses = [], []
        for epoch in range(self.epochs):
            epoch_train_losses = []
            for X_train, y_train in self.ds.get_train_ds():
                y_hat = self.model(X_train)
                train_loss = self.mse_loss(y_hat, y_train)

                grad = gradient(y_hat, y_train, X_train)
                grad = grad.reshape(-1, 1)
                
                epoch_train_losses.append(train_loss.item())

                self.model.update(grad * self.lr)
            
            avg_epoch_train_loss = sum(epoch_train_losses) / len(epoch_train_losses)
        
            epoch_test_losses = []
            for X_test, y_test in self.ds.get_test_ds():
                y_hat = self.model(X_test)
                test_loss = self.mse_loss(y_hat, y_test)

                epoch_test_losses.append(test_loss.item())
            
            avg_epoch_test_loss = sum(epoch_test_losses) / len(epoch_test_losses)

            train_losses.append(avg_epoch_train_loss)
            test_losses.append(avg_epoch_test_loss)

        self.plot_losses(train_losses, test_losses)
        print(self.model)
        
    def plot_losses(self, train_losses, test_losses):
        steps_per_epoch = len(self.ds.X_train) // self.ds.batch_size
        steps = [(i + 1) * steps_per_epoch for i in range(len(train_losses))]
        self.fig.add_trace(go.Scatter(x=steps, y=train_losses, mode='lines', name=f'Train Loss {self.ds.batch_size}'))
        self.fig.add_trace(go.Scatter(x=steps, y=test_losses, mode='lines', name=f'Test Loss {self.ds.batch_size}'))
        

eps = 1e-3
degree = 2
coeffs = np.random.rand(degree)
theta = np.random.rand(degree, 1)
func = PolynomialFunction(degree=degree, coeffs=coeffs, eps=eps)
fig = go.Figure()
lr = 1e-3
T = 50000
models = []
batch_sizes = [1, 2, 4, 8, 16, 64]
for batch_size in batch_sizes:
    model = Hypothesis(degree=degree, theta=np.copy(theta), track_params=True)
    ds = TrainTestDataset(func=func, batch_size=batch_size, eps=eps)
    train_size = len(ds.X_train)
    steps_per_epoch = train_size // batch_size
    epochs = T // steps_per_epoch
    trainer = Trainer(model=model, ds=ds, fig=fig, epochs=epochs, lr=lr)
    trainer.train()
    models.append(model)

fig.show()

Hypothesis(degree=2, theta=[[0.79172493]
 [0.52890552]])
Hypothesis(degree=2, theta=[[0.79172249]
 [0.52890368]])
Hypothesis(degree=2, theta=[[0.79171356]
 [0.52892288]])
Hypothesis(degree=2, theta=[[0.79172622]
 [0.5288928 ]])
Hypothesis(degree=2, theta=[[0.79172378]
 [0.52889661]])
Hypothesis(degree=2, theta=[[0.79172704]
 [0.52889341]])


### Plotting the loss surface


In [132]:
# ── Site palette ────────────────────────────────────────────────────────────
SITE = dict(
    bg_primary   = '#213636',   # plot background
    bg_secondary = '#1a2d2d',   # paper / outer background
    bg_tertiary  = '#2a4444',   # subtle element fill
    border       = '#3a5454',   # grid lines / borders
    text_primary = '#c8c4b8',   # axis labels, legend text
    text_secondary = '#a09890', # tick labels
    accent       = '#bdb76b',   # highlight colour (dark khaki)
    olive        = '#828631',   # secondary accent
)

# Custom colorscale: bg_secondary → bg_tertiary → olive → accent
SITE_COLORSCALE = [
    [0.0,  '#1a2d2d'],
    [0.25, '#2a4444'],
    [0.5,  '#3a5454'],
    [0.75, '#828631'],
    [1.0,  '#bdb76b'],
]

In [136]:
def plot_contour_with_trajectory(dataset, all_models, all_batch_sizes, n_steps=100, pad=0.05):
    X, y = dataset.X_train, dataset.y_train

    # --- bounding box from all trajectory points across all models ---
    all_traj = np.concatenate([np.array(m.trajectory).squeeze() for m in all_models], axis=0)
    t0_min, t0_max = all_traj[:, 0].min(), all_traj[:, 0].max()
    t1_min, t1_max = all_traj[:, 1].min(), all_traj[:, 1].max()
    t0_pad = (t0_max - t0_min) * pad
    t1_pad = (t1_max - t1_min) * pad
    t0_vals = np.linspace(t0_min - t0_pad, t0_max + t0_pad, n_steps)
    t1_vals = np.linspace(t1_min - t1_pad, t1_max + t1_pad, n_steps)

    # --- compute loss grid ---
    loss_grid = np.zeros((n_steps, n_steps))
    for i, t0 in enumerate(t0_vals):
        for j, t1 in enumerate(t1_vals):
            theta_p = np.array([[t0], [t1]])
            y_hat = X @ theta_p
            loss_grid[j, i] = np.mean((y - y_hat) ** 2)

    log_loss = np.log(loss_grid + 1e-12)

    fig = go.Figure()

    # 3D loss surface — site-themed colorscale
    fig.add_trace(go.Surface(
        z=log_loss, x=t0_vals, y=t1_vals,
        opacity=0.80,
        colorscale=SITE_COLORSCALE,
        colorbar=dict(
            title=dict(text='log(MSE)', font=dict(color=SITE['text_primary'])),
            tickfont=dict(color=SITE['text_secondary']),
            bgcolor=SITE['bg_secondary'],
            bordercolor=SITE['border'],
        ),
        name='Loss landscape',
        showscale=True,
    ))

    # 3D trajectory lines
    colors = px.colors.qualitative.Plotly
    for idx, (m, bs) in enumerate(zip(all_models, all_batch_sizes)):
        traj = np.array(m.trajectory).squeeze()[:3000]
        step = max(1, len(traj) // 500)
        traj = traj[::step]
        color = colors[idx % len(colors)]

        traj_z = []
        for t in traj:
            y_hat = X @ t.reshape(-1, 1)
            traj_z.append(np.log(np.mean((y - y_hat) ** 2) + 1e-12))

        fig.add_trace(go.Scatter3d(
            x=traj[:, 0], y=traj[:, 1], z=traj_z,
            mode='lines',
            line=dict(color=color, width=4),
            name=f'bs={bs}',
        ))

    fig.update_traces(
        contours_z=dict(show=True, usecolormap=True,
                        highlightcolor=SITE['accent'], project_z=True),
        selector=dict(type='surface'),
    )

    # Shared axis style
    axis_style = dict(
        backgroundcolor=SITE['bg_tertiary'],
        gridcolor=SITE['border'],
        showbackground=True,
        zerolinecolor=SITE['border'],
        tickfont=dict(color=SITE['text_secondary']),
        title_font=dict(color=SITE['text_primary']),
    )

    fig.update_layout(
        title=dict(
            text='Interative Loss Surface with Training Trajectories (θ₀ vs θ₁)',
            font=dict(color=SITE['text_primary']),
        ),
        paper_bgcolor=SITE['bg_secondary'],
        scene=dict(
            xaxis=dict(**axis_style, title='θ₀'),
            yaxis=dict(**axis_style, title='θ₁'),
            zaxis=dict(**axis_style, title='log(MSE)'),
            bgcolor=SITE['bg_primary'],
        ),
        legend=dict(
            orientation='h',
            x=0.5,
            y=-0.12,
            xanchor='center',
            yanchor='top',
            font=dict(color=SITE['text_primary']),
            bgcolor=SITE['bg_secondary'],
            bordercolor=SITE['border'],
        ),
        autosize=True,
        margin=dict(l=0, r=0, t=40, b=0),
    )

    fig.write_html(
        PLOTS_DIR / "gradient_descent.html",
        config=dict(responsive=True, displayModeBar=True),
        include_plotlyjs='cdn',
    )
    fig.show()

plot_contour_with_trajectory(ds, models, batch_sizes)